In [ ]:
!nvidia-smi

Mon May 11 18:11:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile vectoradd.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// CUDA Kernel
__global__ void vectorAdd(int *A, int *B, int *C, int N)
{
    int i = threadIdx.x;

    if(i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    int N;

    cout << "Enter size: ";
    cin >> N;

    int size = N * sizeof(int);

    // Host arrays
    int *h_A = new int[N];
    int *h_B = new int[N];
    int *h_C = new int[N];

    cout << "Enter vector A:\n";
    for(int i = 0; i < N; i++)
        cin >> h_A[i];

    cout << "Enter vector B:\n";
    for(int i = 0; i < N; i++)
        cin >> h_B[i];

    // Device arrays
    int *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy data to GPU
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Launch kernel
    vectorAdd<<<1, N>>>(d_A, d_B, d_C, N);

    // Wait for GPU
    cudaDeviceSynchronize();

    // Copy result back
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    cout << "\nResultant Vector:\n";

    for(int i = 0; i < N; i++)
        cout << h_C[i] << " ";

    cout << endl;

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    return 0;
}

Writing vectoradd.cu


In [ ]:
!nvcc vectoradd.cu -o vectoradd

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vectoradd

Enter size: 5
Enter vector A:
234 382 122 982 121
Enter vector B:
982 132 182 102 823

Resultant Vector:
1216 514 304 1084 944 
